## Imports

In [53]:
from pathlib import Path
import json
import joblib
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from scipy import sparse

RANDOM_STATE = 42

##  Project root

In [ ]:
from pathlib import Path

# ============================================================
# PROJECT PATH CONFIGURATION
# ============================================================

PROJECT_ROOT = Path("/home/tonoy-sen/Desktop/pro/MlProjects/multidisease-risk-profiler")

DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed" / "ml_ready"

# Input cleaned dataset
DATA_PATH = DATA_DIR  / "processed" / "cleaned_dataset.csv"


# Model / preprocessing directory
MODEL_DIR = PROJECT_ROOT / "ml-service" / "models" / "preprocessors"

# Create required directories
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

# Print paths
print("Project root:", PROJECT_ROOT)
print("Input:", DATA_PATH)
print("ML-ready output:", PROCESSED_DIR)
print("Preprocessor output:", MODEL_DIR)

Project root: /home/tonoy-sen/Desktop/pro/MlProjects/multidisease-risk-profiler
Input: /home/tonoy-sen/Desktop/pro/MlProjects/multidisease-risk-profiler/data/processed/cleaned_dataset.csv
ML-ready output: /home/tonoy-sen/Desktop/pro/MlProjects/multidisease-risk-profiler/data/processed/ml_ready
Preprocessor output: /home/tonoy-sen/Desktop/pro/MlProjects/multidisease-risk-profiler/ml-service/models/preprocessors


## Load cleaned dataset

In [55]:
df = pd.read_csv(DATA_PATH)
df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_")

for col in df.select_dtypes(include="object").columns:
    df[col] = df[col].astype("string").str.strip()

print("Shape:", df.shape)
display(df.head())

/tmp/ipykernel_37198/1055819966.py:4: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in df.select_dtypes(include="object").columns:


Shape: (278701, 39)


,composite_key,age_level,gender,bmi_level,smoking,diabetes,age,age_normalized,bmi,hypertension,...,salt_intake,heart_rate,hdl,ldl,education_level,employment_status,source_dataset,disease_flags,sublabel,label
0,Young_Female_Normal_Never_No,Young,Female,Normal,Never,No,9,0.080460,19.20,0.0,...,8.685304,74.329073,65.651757,129.220447,Primary,Retired,diabetes,"0,0,0",N,Normal
1,Young_Female_Normal_Never_No,Young,Female,Normal,Never,No,3,0.011494,22.55,0.0,...,8.685304,74.329073,65.651757,129.220447,Primary,Retired,diabetes,"0,0,0",N,Normal
2,Young_Female_Normal_Never_No,Young,Female,Normal,Never,No,3,0.011494,22.89,0.0,...,8.685304,74.329073,65.651757,129.220447,Primary,Retired,diabetes,"0,0,0",N,Normal
3,Young_Female_Normal_Never_No,Young,Female,Normal,Never,No,3,0.011494,23.12,0.0,...,8.685304,74.329073,65.651757,129.220447,Primary,Retired,diabetes,"0,0,0",N,Normal
4,Young_Female_Normal_Never_No,Young,Female,Normal,Never,No,4,0.022989,19.61,0.0,...,8.685304,74.329073,65.651757,129.220447,Primary,Retired,diabetes,"0,0,0",N,Normal


## Create targets from confirmed disease_flags mapping.
### disease_flags = Diabetes, Hypertension, Heart Disease

In [56]:
def parse_flags(value):
    if pd.isna(value):
        return (np.nan, np.nan, np.nan)
    bits = str(value).strip().replace(" ", "")
    if bits not in {
        "0,0,0", "0,0,1", "0,1,0", "0,1,1",
        "1,0,0", "1,0,1", "1,1,0", "1,1,1"
    }:
        return (np.nan, np.nan, np.nan)
    return tuple(map(int, bits.split(",")))

flags = df["disease_flags"].apply(parse_flags)
df["diabetes_target"] = flags.apply(lambda x: x[0])
df["hypertension_target"] = flags.apply(lambda x: x[1])
df["heart_disease_target"] = flags.apply(lambda x: x[2])
df["obesity_target"] = df["bmi_level"].astype("string").str.strip()

target_cols = [
    "diabetes_target",
    "hypertension_target",
    "heart_disease_target",
    "obesity_target",
]

## Validate targets

In [57]:
for col in target_cols:
    print("\n", col)
    print(df[col].value_counts(dropna=False).sort_index())

# Diabetes consistency check
if "diabetes" in df.columns:
    original = df["diabetes"].map({"Yes": 1, "No": 0})
    print("\nDiabetes mismatches:",
          (original != df["diabetes_target"]).sum())

# Remove rows missing any of the four targets
df_model = df.dropna(subset=target_cols).copy()
print("\nRows before:", len(df))
print("Rows after :", len(df_model))



 diabetes_target
diabetes_target
0    178083
1    100618
Name: count, dtype: int64

 hypertension_target
hypertension_target
0    272810
1      5891
Name: count, dtype: int64

 heart_disease_target
heart_disease_target
0    140532
1    138169
Name: count, dtype: int64

 obesity_target
obesity_target
Normal         70591
Obese          98377
Overweight     76897
Underweight    32836
Name: count, dtype: Int64

Diabetes mismatches: 0

Rows before: 278701
Rows after : 278701


## Remove metadata, leakage, and redundant columns

In [58]:
GLOBAL_EXCLUDE = {
    "composite_key",
    "source_dataset",
    "disease_flags",
    "sublabel",
    "label",
    "diabetes",
    "hypertension",
    "heart_disease",
    "age_normalized",
    "bmi_level",
    "diabetes_target",
    "hypertension_target",
    "heart_disease_target",
    "obesity_target",
}

common_features = [
    c for c in df_model.columns if c not in GLOBAL_EXCLUDE
]

FEATURES = {
    "diabetes": common_features.copy(),
    "hypertension": common_features.copy(),
    "heart_disease": common_features.copy(),
    "obesity": [c for c in common_features if c != "bmi"],
}

for task, cols in FEATURES.items():
    print(f"\n{task.upper()}: {len(cols)} features")
    print(cols)
    print("Leakage overlap:", GLOBAL_EXCLUDE.intersection(cols))




DIABETES: 29 features
['age_level', 'gender', 'smoking', 'age', 'bmi', 'hba1c_level', 'glucose', 'cholesterol', 'sleep_hours', 'triglycerides', 'physical_activity', 'family_history', 'stress_level', 'low_hdl_cholesterol', 'high_ldl_cholesterol', 'blood_pressure', 'high_blood_pressure', 'sugar_consumption', 'crp_level', 'homocysteine_level', 'systolic_bp', 'diastolic_bp', 'alcohol_intake', 'salt_intake', 'heart_rate', 'hdl', 'ldl', 'education_level', 'employment_status']
Leakage overlap: set()

HYPERTENSION: 29 features
['age_level', 'gender', 'smoking', 'age', 'bmi', 'hba1c_level', 'glucose', 'cholesterol', 'sleep_hours', 'triglycerides', 'physical_activity', 'family_history', 'stress_level', 'low_hdl_cholesterol', 'high_ldl_cholesterol', 'blood_pressure', 'high_blood_pressure', 'sugar_consumption', 'crp_level', 'homocysteine_level', 'systolic_bp', 'diastolic_bp', 'alcohol_intake', 'salt_intake', 'heart_rate', 'hdl', 'ldl', 'education_level', 'employment_status']
Leakage overlap: set(

## Split function: 70/15/15 with stratification

In [59]:
def split_dataset(data, features, target):
    X = data[features].copy()
    y = data[target].copy()

    X_train, X_temp, y_train, y_temp = train_test_split(
        X, y,
        test_size=0.30,
        random_state=RANDOM_STATE,
        stratify=y,
    )

    X_val, X_test, y_val, y_test = train_test_split(
        X_temp, y_temp,
        test_size=0.50,
        random_state=RANDOM_STATE,
        stratify=y_temp,
    )

    return X_train, X_val, X_test, y_train, y_val, y_test


## Preprocessor: fit only on training data

In [60]:
def make_preprocessor(X):
    numeric = X.select_dtypes(include=["number"]).columns.tolist()
    categorical = X.select_dtypes(
        include=["object", "string", "category", "bool"]
    ).columns.tolist()

    num_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ])

    cat_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=True)),
    ])

    transformers = []
    if numeric:
        transformers.append(("num", num_pipe, numeric))
    if categorical:
        transformers.append(("cat", cat_pipe, categorical))

    return ColumnTransformer(transformers=transformers, remainder="drop"), numeric, categorical


## Prepare and save four ML datasets

In [61]:
results = {}

for task, features in FEATURES.items():
    target = f"{task}_target"

    print("\n" + "_" * 80+"\n")
    print("TASK:", task.upper())

    X_train, X_val, X_test, y_train, y_val, y_test = split_dataset(
        df_model, features, target
    )

    preprocessor, numeric, categorical = make_preprocessor(X_train)

    X_train_p = preprocessor.fit_transform(X_train)
    X_val_p = preprocessor.transform(X_val)
    X_test_p = preprocessor.transform(X_test)

    print("Raw:", X_train.shape, X_val.shape, X_test.shape)
    print("Processed:", X_train_p.shape, X_val_p.shape, X_test_p.shape)

    np.save(PROCESSED_DIR / f"{task}_X_train.npy", X_train_p)
    np.save(PROCESSED_DIR / f"{task}_X_val.npy", X_val_p)
    np.save(PROCESSED_DIR / f"{task}_X_test.npy", X_test_p)

    y_train.to_csv(PROCESSED_DIR / f"{task}_y_train.csv", index=False)
    y_val.to_csv(PROCESSED_DIR / f"{task}_y_val.csv", index=False)
    y_test.to_csv(PROCESSED_DIR / f"{task}_y_test.csv", index=False)

    # Save raw splits for inspection / explainability
    X_train.to_csv(PROCESSED_DIR / f"{task}_X_train_raw.csv", index=False)
    X_val.to_csv(PROCESSED_DIR / f"{task}_X_val_raw.csv", index=False)
    X_test.to_csv(PROCESSED_DIR / f"{task}_X_test_raw.csv", index=False)

    joblib.dump(preprocessor, MODEL_DIR / f"{task}_preprocessor.joblib")

    metadata = {
        "task": task,
        "target": target,
        "raw_features": features,
        "numeric_features": numeric,
        "categorical_features": categorical,
        "train_rows": len(X_train),
        "validation_rows": len(X_val),
        "test_rows": len(X_test),
        "processed_features": X_train_p.shape[1],
        "random_state": RANDOM_STATE,
    }

    with open(PROCESSED_DIR / f"{task}_metadata.json", "w", encoding="utf-8") as f:
        json.dump(metadata, f, indent=2)

    results[task] = metadata



________________________________________________________________________________

TASK: DIABETES
Raw: (195090, 29) (41805, 29) (41806, 29)
Processed: (195090, 49) (41805, 49) (41806, 49)

________________________________________________________________________________

TASK: HYPERTENSION
Raw: (195090, 29) (41805, 29) (41806, 29)
Processed: (195090, 49) (41805, 49) (41806, 49)

________________________________________________________________________________

TASK: HEART_DISEASE
Raw: (195090, 29) (41805, 29) (41806, 29)
Processed: (195090, 49) (41805, 49) (41806, 49)

________________________________________________________________________________

TASK: OBESITY
Raw: (195090, 28) (41805, 28) (41806, 28)
Processed: (195090, 48) (41805, 48) (41806, 48)


## Verify class proportions

In [62]:
for task in FEATURES:
    target = f"{task}_target"
    print("\n" + "_" * 60 + "\n")
    print(task.upper())

    for split in ["train", "val", "test"]:
        y = pd.read_csv(PROCESSED_DIR / f"{task}_y_{split}.csv")[target]
        print(f"\n{split}:")
        print(y.value_counts(normalize=True).sort_index())


____________________________________________________________

DIABETES

train:
diabetes_target
0    0.638977
1    0.361023
Name: proportion, dtype: float64

val:
diabetes_target
0    0.638967
1    0.361033
Name: proportion, dtype: float64

test:
diabetes_target
0    0.638975
1    0.361025
Name: proportion, dtype: float64

____________________________________________________________

HYPERTENSION

train:
hypertension_target
0    0.978861
1    0.021139
Name: proportion, dtype: float64

val:
hypertension_target
0    0.978878
1    0.021122
Name: proportion, dtype: float64

test:
hypertension_target
0    0.978855
1    0.021145
Name: proportion, dtype: float64

____________________________________________________________

HEART_DISEASE

train:
heart_disease_target
0    0.504239
1    0.495761
Name: proportion, dtype: float64

val:
heart_disease_target
0    0.504246
1    0.495754
Name: proportion, dtype: float64

test:
heart_disease_target
0    0.504234
1    0.495766
Name: proportion, dtype: 

## Final summary

In [63]:
summary = pd.DataFrame([
    {
        "task": task,
        "target": meta["target"],
        "raw_features": len(meta["raw_features"]),
        "processed_features": meta["processed_features"],
        "train_rows": meta["train_rows"],
        "validation_rows": meta["validation_rows"],
        "test_rows": meta["test_rows"],
    }
    for task, meta in results.items()
])

display(summary)

print("\nML DATA PREPARATION COMPLETE.")
print("Processed data:", PROCESSED_DIR)
print("Preprocessors:", MODEL_DIR)


,task,target,raw_features,processed_features,train_rows,validation_rows,test_rows
0,diabetes,diabetes_target,29,49,195090,41805,41806
1,hypertension,hypertension_target,29,49,195090,41805,41806
2,heart_disease,heart_disease_target,29,49,195090,41805,41806
3,obesity,obesity_target,28,48,195090,41805,41806



ML DATA PREPARATION COMPLETE.
Processed data: /home/tonoy-sen/Desktop/pro/MlProjects/multidisease-risk-profiler/data/processed/ml_ready
Preprocessors: /home/tonoy-sen/Desktop/pro/MlProjects/multidisease-risk-profiler/ml-service/models/preprocessors
